In [24]:
# --- 1. Install the CBC Solver (Essential for Google Colab/Ubuntu-based environments) ---
# Run this cell first if you are in Google Colab or a similar environment.
# This ensures that PuLP can find and use the necessary solver.
!sudo apt-get install coinor-cbc

# --- 2. Import Necessary Libraries ---
import pulp
import pandas as pd
from io import StringIO

# --- 3. Configuration and Debugging Flags ---
# Set this to True if you want to enforce the LiJ (Lower Bound) constraints on arcs.
# If your problem is Infeasible, try setting this to False first to see if LiJ constraints are the cause.
ENABLE_LIJ_CONSTRAINTS = False # Keep as False for now to check feasibility with updated data

# --- 4. Raw Data Table (UPDATED with corrected warehouse-to-demand connections) ---
# The 'From' field for rows 52, 53, 56, 57, 60, and 61 has been updated
# from district names to their respective warehouse names to define new arcs.
raw_data = """ID	From	To	"Distance (km)"	"Unit Cost (Rwfs/km/T)"	"Total Cost (Rwfs/ T)"	"Truck capacity (Tons/year)"	"LiJ- Lower bound per trip (Tons)"	"LiJ- Lower bound (Tons/ year)"		"Supply capacity (Tons/Year)"	"Demand (Tons/year)"	UIJ (upper bound)
1	Nyagatare	Kayonza 1	86.8	200	17360	4160	0.58	59.9		112485.2434	4500	4160
2	Nyagatare	Kayonza 2	106	200	21200	4160	0.47	49.1		112485.2434	4500	4160
3	Nyagatare	Rwamagana-Musha	110	200	22000	4160	0.45	47.3		112485.2434	1500	1500
4	Nyagatare	Bugesera-Mayange	173	200	34600	4160	0.29	30.1		112485.2434	12000	4160
5	Nyagatare	Gasabo	132	200	26400	4160	0.38	39.4		112485.2434	23525.0671	4160
6	Nyagatare	Kicukiro	137	200	27400	4160	0.36	38.0		112485.2434	14983.11202	4160
7	Nyagatare	Nyarugenge	134	200	26800	4160	0.37	38.8		112485.2434	11837.37898	4160
8	Kirehe	Kayonza 1	61.4	200	12280	4160	0.81	84.7		48783.3388	4500	4160
9	Kirehe	Kayonza 2	62.4	200	12480	4160	0.80	83.3		48783.3388	4500	4160
10	Kirehe	Rwamagana-Musha	98.8	200	19760	4160	0.51	52.6		48783.3388	1500	1500
11	Kirehe	Bugesera-Mayange	104	200	20800	4160	0.48	50.0		48783.3388	12000	4160
12	Kirehe	Gasabo	145	200	29000	4160	0.34	35.9		48783.3388	23525.0671	4160
13	Kirehe	Kicukiro	138	200	27600	4160	0.36	37.7		48783.3388	14983.11202	4160
14	Kirehe	Nyarugenge	145	200	29000	4160	0.34	35.9		48783.3388	11837.37898	4160
15	Rwamagana	Kayonza 1	43.7	200	8740	4160	1.14	119.0		5020.96474	4500	4160
16	Rwamagana	Kayonza 2	63.1	200	12620	4160	0.79	82.4		5020.96474	4500	4160
17	Rwamagana	Rwamagana-Musha	14.1	200	2820	4160	3.55	368.8		5020.96474	1500	1500
18	Rwamagana	Bugesera-Mayange	66.7	200	13340	4160	0.75	78.0		5020.96474	12000	4160
19	Rwamagana	Gasabo	38.3	200	7660	4160	1.31	135.8		5020.96474	23525.0671	4160
20	Rwamagana	Kicukiro	31.5	200	6300	4160	1.59	165.1		5020.96474	14983.11202	4160
21	Rwamagana	Nyarugenge	38	200	7600	4160	1.32	136.8		5020.96474	11837.37898	4160
22	Kayonza	Kayonza 1	11.7	200	2340	4160	4.27	444.4		25981.53448	4500	4160
23	Kayonza	Kayonza 2	7.7	200	1540	4160	6.49	675.3		25981.53448	4500	4160
24	Kayonza	Rwamagana-Musha	49.1	200	9820	4160	1.02	105.9		25981.53448	1500	1500
25	Kayonza	Bugesera-Mayange	85.4	200	17080	4160	0.59	60.9		25981.53448	12000	4160
26	Kayonza	Gasabo	95.5	200	19100	4160	0.52	54.5		25981.53448	23525.0671	4160
27	Kayonza	Kicukiro	88.8	200	17760	4160	0.56	58.6		25981.53448	14983.11202	4160
28	Kayonza	Nyarugenge	95.2	200	19040	4160	0.53	54.6		25981.53448	11837.37898	4160
29	Bugesera	Kayonza 1	98.9	200	19780	4160	0.51	52.6		1596.93174	4500	1597
30	Bugesera	Kayonza 2	103	200	20600	4160	0.49	50.5		1596.93174	4500	1597
31	Bugesera	Rwamagana-Musha	60.7	200	12140	4160	0.82	85.7		1596.93174	1500	1500
32	Bugesera	Bugesera-Mayange	11	200	2200	4160	4.55	472.7		1596.93174	12000	1597
33	Bugesera	Gasabo	33.2	200	6640	4160	1.51	156.6		1596.93174	23525.0671	1597
34	Bugesera	Kicukiro	25.4	200	5080	4160	1.97	204.7		1596.93174	14983.11202	1597
35	Bugesera	Nyarugenge	32.9	200	6580	4160	1.52	158.1		1596.93174	11837.37898	1597
36	Ngoma	Kayonza 1	27.7	200	5540	4160	1.81	187.7		19377.61984	4500	4160
37	Ngoma	Kayonza 2	27.8	200	5560	4160	1.80	187.1		19377.61984	4500	4160
38	Ngoma	Rwamagana-Musha	65.1	200	13020	4160	0.77	79.9		19377.61984	1500	1500
39	Ngoma	Bugesera-Mayange	64.9	200	12980	4160	0.77	80.1		19377.61984	12000	4160
40	Ngoma	Gasabo	111	200	22200	4160	0.45	46.8		19377.61984	23525.0671	4160
41	Ngoma	Kicukiro	105	200	21000	4160	0.48	49.5		19377.61984	14983.11202	4160
42	Ngoma	Nyarugenge	111	200	22200	4160	0.45	46.8		19377.61984	11837.37898	4160
43	Gatsibo	Kayonza 1	52.3	200	10460	4160	0.96	99.4		35510.77112	4500	4160
44	Gatsibo	Kayonza 2	71.6	200	14320	4160	0.70	72.6		35510.77112	4500	4160
45	Gatsibo	Rwamagana-Musha	75.8	200	15160	4160	0.66	68.6		35510.77112	1500	1500
46	Gatsibo	Bugesera-Mayange	145	200	29000	4160	0.34	35.9		35510.77112	12000	4160
47	Gatsibo	Gasabo	104	200	20800	4160	0.48	50.0		35510.77112	23525.0671	4160
48	Gatsibo	Kicukiro	115	200	23000	4160	0.43	45.2		35510.77112	14983.11202	4160
49	Gatsibo	Nyarugenge	107	200	21400	4160	0.47	48.6		35510.77112	11837.37898	4160
50	Kayonza 1	Gasabo	83.8	200	16760	4160	0.60	62.1		4500	23525.0671	4160
51	Kayonza 2	Gasabo	103	200	20600	4160	0.49	50.5		4500	23525.0671	4160
52	Rwamagana-Musha	Gasabo	51.9	200	10380	4160	0.96	100.2		1500	23525.0671	1500
53	Bugesera-Mayange	Gasabo	44.1	200	8820	4160	1.13	117.9		12000	23525.0671	4160
54	Kayonza 1	Kicukiro	77.1	200	15420	4160	0.65	67.4		4500	14983.11202	4160
55	Kayonza 2	Kicukiro	96.5	200	19300	4160	0.52	53.9		4500	14983.11202	4160
56	Rwamagana-Musha	Kicukiro	45.2	200	9040	4160	1.11	115.0		1500	14983.11202	1500
57	Bugesera-Mayange	Kicukiro	36.4	200	7280	4160	1.37	142.9		12000	14983.11202	1597
58	Kayonza 1	Nyarugenge	83.4	200	16680	4160	0.60	62.4		4500	11837.37898	4160
59	Kayonza 2	Nyarugenge	103	200	20600	4160	0.49	50.5		4500	11837.37898	4160
60	Rwamagana-Musha	Nyarugenge	51.5	200	10300	4160	0.97	101.0		1500	11837.37898	1500
61	Bugesera-Mayange	Nyarugenge	43.7	200	8740	4160	1.14	119.0		12000	11837.37898	1597
"""

# Read the data into a pandas DataFrame for easier processing
df = pd.read_csv(StringIO(raw_data), sep='\t')

# Clean column names (remove quotes and extra spaces)
df.columns = df.columns.str.replace('"', '').str.strip()

# --- 5. Node Name Mapping for Consistency ---
# Standardize names to avoid typos and ensure unique identification in the model.
node_name_map = {
    'Nyagatare': 'Nyagatare_D',       # District/Supply Node
    'Kirehe': 'Kirehe_D',             # District/Supply Node
    'Rwamagana': 'Rwamagana_D',       # District/Supply Node
    'Kayonza': 'Kayonza_D',           # District/Supply Node
    'Bugesera': 'Bugesera_D',         # District/Supply Node
    'Ngoma': 'Ngoma_D',               # District/Supply Node
    'Gatsibo': 'Gatsibo_D',           # District/Supply Node
    'Kayonza 1': 'Kayonza_W1',        # Warehouse 1
    'Kayonza 2': 'Kayonza_W2',        # Warehouse 2
    'Rwamagana-Musha': 'Rwamagana_WM',# Rwamagana Warehouse
    'Bugesera-Mayange': 'Bugesera_WM',# Bugesera Warehouse
    'Gasabo': 'Gasabo_Dem',           # Demand Destination
    'Kicukiro': 'Kicukiro_Dem',       # Demand Destination
    'Nyarugenge': 'Nyarugenge_Dem'    # Demand Destination
}

# Apply the mapping to 'From' and 'To' columns in the DataFrame
df['From'] = df['From'].replace(node_name_map)
df['To'] = df['To'].replace(node_name_map)

# --- 6. Data Extraction for Model Parameters ---

# Initialize dictionaries to hold unique node properties
supply_nodes = {}         # {supply_node_name: capacity}
demand_nodes = {}         # {demand_node_name: requirement}
eax_warehouse_capacities = {} # {warehouse_node_name: throughput_capacity}

# Arc-specific data
costs = {}                # {from_node: {to_node: cost}}
arc_capacities = {}       # {from_node: {to_node: UIJ_upper_bound}}
arc_lower_bounds = {}     # {from_node: {to_node: LiJ_lower_bound}}
all_nodes = set()         # Collects all unique nodes in the network

# Populate data structures by iterating through the DataFrame
for index, row in df.iterrows():
    f = row['From']
    t = row['To']

    # Populate supply_nodes (only for nodes ending with '_D' and appearing as 'From')
    if f.endswith('_D') and f not in supply_nodes:
        supply_nodes[f] = row['Supply capacity (Tons/Year)']

    # Populate demand_nodes (only for nodes ending with '_Dem' and appearing as 'To')
    if t.endswith('_Dem') and t not in demand_nodes:
        demand_nodes[t] = row['Demand (Tons/year)']

    # Populate eax_warehouse_capacities (for specific warehouse names appearing as 'To')
    if (t.startswith('Kayonza_W') or t.endswith('_WM')) and t not in eax_warehouse_capacities:
        # In your data, the 'Demand (Tons/year)' column seems to represent warehouse capacity
        # when a warehouse is a 'To' node.
        eax_warehouse_capacities[t] = row['Demand (Tons/year)']

    # Populate costs, arc_capacities (UIJ), and arc_lower_bounds (LiJ)
    if f not in costs:
        costs[f] = {}
    costs[f][t] = row['Total Cost (Rwfs/ T)']

    if f not in arc_capacities:
        arc_capacities[f] = {}
    arc_capacities[f][t] = row['UIJ (upper bound)']

    if f not in arc_lower_bounds:
        arc_lower_bounds[f] = {}
    arc_lower_bounds[f][t] = row['LiJ- Lower bound (Tons/ year)']

    # Add both 'From' and 'To' nodes to the set of all nodes in the network
    all_nodes.add(f)
    all_nodes.add(t)

# --- 7. Identify Node Categories for Model Formulation ---
supply_districts = list(supply_nodes.keys())
demand_districts = list(demand_nodes.keys())
warehouses = list(eax_warehouse_capacities.keys())

# Transshipment nodes are any nodes that are not purely a supply district or a demand district.
# They act as intermediate points where flow can arrive and then depart.
transshipment_nodes = [node for node in all_nodes if node not in supply_districts and node not in demand_districts]


# --- 8. Initial Feasibility Check: Total Supply vs. Total Demand ---
# This is crucial for diagnosing infeasibility before running the solver.
total_supply_available = sum(supply_nodes.values())
total_demand_required = sum(demand_nodes.values())

print(f"--- Initial Feasibility Check ---")
print(f"Total Supply Available: {total_supply_available:,.2f} Tons")
print(f"Total Demand Required: {total_demand_required:,.2f} Tons")

if total_supply_available < total_demand_required:
    print("\nWARNING: Total supply is LESS than total demand. The problem is inherently INFEASIBLE.")
    print("         You cannot meet all demand with the current supply levels.")
elif total_supply_available == total_demand_required:
    print("\nTotal supply EQUALS total demand. The problem is balanced, but may still be infeasible due to routing/capacities.")
else:
    print("\nTotal supply is GREATER than total demand. The problem should be able to meet demand, assuming sufficient capacities.")
print("---------------------------------\n")


# --- 9. Create the LP Problem ---
# Define the optimization problem: Minimize total transportation cost.
prob = pulp.LpProblem("MaizeDistributionNetwork", pulp.LpMinimize)

# --- 10. Define Decision Variables ---
# 'flow[(f, t)]' represents the amount of maize (in Tons/Year) transported from node 'f' to node 't'.
flow = pulp.LpVariable.dicts("Flow",
                             ((f, t) for f in all_nodes for t in all_nodes if f in costs and t in costs[f]),
                             lowBound=0,  # Flow cannot be negative
                             cat='Continuous') # Flow can be any non-negative real number

# --- 11. Objective Function ---
# Minimize the total transportation cost, which is the sum of (flow * cost) for all arcs.
prob += pulp.lpSum(flow[f, t] * costs[f][t] for f, t in flow), "Total_Transportation_Cost"

# --- 12. Add Constraints to the Problem ---

# a. Supply Constraints: Total outflow from each supply district cannot exceed its supply capacity.
for s in supply_districts:
    # Ensure the sum only considers arcs that actually exist in the 'flow' variables
    outgoing_arcs = [(f_key, t) for f_key, t in flow if f_key == s]
    if outgoing_arcs: # Only add constraint if there are outgoing arcs
        prob += pulp.lpSum(flow[s, t] for s, t in outgoing_arcs) <= supply_nodes.get(s, 0), f"Supply_Limit_{s}"
    else:
        print(f"Warning: Supply district {s} has no outgoing arcs defined in the data. Its supply cannot be utilized.")

# b. Demand Constraints: Total inflow to each demand district must meet its demand requirement.
for d in demand_districts:
    # Ensure the sum only considers arcs that actually exist in the 'flow' variables
    incoming_arcs = [(f, t_key) for f, t_key in flow if t_key == d]
    if incoming_arcs: # Only add constraint if there are incoming arcs
        prob += pulp.lpSum(flow[f, d] for f, d in incoming_arcs) >= demand_nodes.get(d, 0), f"Demand_Fulfillment_{d}"
    else:
        print(f"Warning: Demand district {d} has no incoming arcs defined in the data. Its demand cannot be met.")


# c. Flow Conservation at Transshipment Nodes: Inflow must equal outflow for intermediate nodes.
for node in transshipment_nodes:
    incoming_flow_sum = pulp.lpSum(flow[f, node] for f, t in flow if t == node)
    outgoing_flow_sum = pulp.lpSum(flow[node, t] for f, t in flow if f == node)
    prob += incoming_flow_sum == outgoing_flow_sum, f"Flow_Conservation_{node}"

# d. Arc Capacity Constraints (UIJ - Upper Bound for each link):
#    The flow on any arc cannot exceed its specified upper capacity.
for f, t in flow:
    if f in arc_capacities and t in arc_capacities[f]:
        prob += flow[f, t] <= arc_capacities[f][t], f"Arc_Capacity_UIJ_{f}_to_{t}"
    else:
        # Fallback for data inconsistencies: assign a very high capacity if not found.
        # This warning helps identify missing data points.
        print(f"Warning: UIJ Arc capacity not found for {f} to {t}. Setting a very high upper bound.")
        prob += flow[f, t] <= 10**9

# e. Arc Lower Bound Constraints (LiJ - Lower Bound for each link):
#    The flow on any arc must be at least its specified lower bound (if ENABLE_LIJ_CONSTRAINTS is True).
if ENABLE_LIJ_CONSTRAINTS:
    for f, t in flow:
        if f in arc_lower_bounds and t in arc_lower_bounds[f]:
            lij_val = arc_lower_bounds[f][t]
            if lij_val > 0: # Only add constraint if the lower bound is meaningful (greater than zero)
                prob += flow[f, t] >= lij_val, f"Arc_Capacity_LIJ_{f}_to_{t}"
        # If LiJ is not found or is 0, the variable's default lowBound=0 applies, which is correct.

# f. Nodal Capacity Constraints for EAX Warehouses (Throughput Capacity):
#    The total flow entering OR leaving a warehouse cannot exceed its capacity.
#    In a transshipment model, usually the *sum* of inflow or outflow (not both) is limited by capacity.
for w in warehouses:
    total_inflow_to_warehouse = pulp.lpSum(flow[f, w] for f, t_key in flow if t_key == w)
    total_outflow_from_warehouse = pulp.lpSum(flow[w, t] for f_key, t in flow if f_key == w)

    # Limiting total inflow to capacity
    prob += total_inflow_to_warehouse <= eax_warehouse_capacities.get(w, 0), f"Warehouse_Inflow_Limit_{w}"
    # Limiting total outflow to capacity (typically inflow or outflow is sufficient, but both ensures robustness)
    prob += total_outflow_from_warehouse <= eax_warehouse_capacities.get(w, 0), f"Warehouse_Outflow_Limit_{w}"


# --- 13. Solve the Problem ---
# Use the COIN_CMD solver, which is robust for continuous linear problems.
# msg=True provides verbose output from the solver, which can be useful for debugging.
print("\n--- Running the Optimization Solver ---")
prob.solve(pulp.COIN_CMD(msg=True))
print("--- Solver Finished ---\n")

# --- 14. Print Results ---
print(f"Problem Status: {pulp.LpStatus[prob.status]}\n")

# Use integer values for comparison to avoid potential AttributeError issues with pulp.LpStatus
if prob.status == 1: # Represents pulp.LpStatus.Optimal
    print(f"Optimal Total Transportation Cost: {pulp.value(prob.objective):,.2f} Rwfs\n")
    print("Flows (Tons/Year):")
    # Sort flows for readability and filter out very small (effectively zero) flows
    sorted_flows = sorted([((f, t), flow[f, t].varValue) for f, t in flow if flow[f, t].varValue is not None and flow[f, t].varValue > 0.001],
                          key=lambda item: item[1], reverse=True)
    for (f, t), val in sorted_flows:
        # Using .get() for robustness in case a cost/capacity isn't found (though it should be by now)
        cost_val = costs.get(f, {}).get(t, 'N/A')
        uij_val = arc_capacities.get(f, {}).get(t, 'N/A')
        lij_val = arc_lower_bounds.get(f, {}).get(t, 'N/A')
        print(f"  {f} -> {t}: {val:,.2f} Tons (Cost: {cost_val:,.2f} Rwfs/T, UIJ: {uij_val:,.2f} Tons, LIJ: {lij_val:,.2f} Tons)")

    print("\nSupply Utilization:")
    for s in supply_districts:
        # Sum outgoing flows from supply nodes
        outgoing_flow = sum(flow[s, t].varValue for f_key, t in flow if f_key == s and flow[s, t].varValue is not None)
        print(f"  {s}: {outgoing_flow:,.2f} / {supply_nodes.get(s, 0):,.2f} Tons supplied")

    print("\nDemand Fulfillment:")
    for d in demand_districts:
        # Sum incoming flows to demand nodes
        incoming_flow = sum(flow[f, d].varValue for f, t_key in flow if t_key == d and flow[f, d].varValue is not None)
        print(f"  {d}: {incoming_flow:,.2f} / {demand_nodes.get(d, 0):,.2f} Tons demanded")

    print("\nWarehouse Throughput:")
    for w in warehouses:
        # Sum incoming and outgoing flows for warehouses
        incoming_flow_w = sum(flow[f, w].varValue for f, t_key in flow if t_key == w and flow[f, w].varValue is not None)
        outgoing_flow_w = sum(flow[w, t].varValue for f_key, t in flow if f_key == w and flow[w, t].varValue is not None)
        print(f"  {w}: In {incoming_flow_w:,.2f} Tons, Out {outgoing_flow_w:,.2f} Tons (Capacity: {eax_warehouse_capacities.get(w, 0):,.2f} Tons)")

elif prob.status == -1: # Represents pulp.LpStatus.Infeasible
    print("The problem is Infeasible. There's no way to satisfy all demand given the current supply and capacities.")
    print("Common reasons and troubleshooting steps:")
    print("1. **Total Supply vs. Total Demand:** Check the 'Initial Feasibility Check' above. If total supply is less than total demand, the problem is impossible.")
    print("2. **Lower Bound (LiJ) Constraints:** These are often the cause of infeasibility. Try setting `ENABLE_LIJ_CONSTRAINTS = False` at the top of the script and re-run.")
    print("   If it becomes feasible, then your LiJ values are too restrictive for the network's capabilities.")
    print("3. **Arc Capacities (UIJ):** Are certain arcs (paths) too small to carry the required flow? Look for bottlenecks.")
    print("4. **Warehouse Capacities:** Are warehouse throughput capacities too low to handle the transshipment volume?")
    print("5. **Network Connectivity:** Ensure every demand point has a viable path from a supply point, considering all capacities.")
elif prob.status == -3: # Represents pulp.LpStatus.Undefined
    print("The problem is Undefined. This can happen if the solver cannot find a solution (e.g., highly degenerate model) or if there are numerical issues.")
else:
    print(f"Problem could not be solved to optimality. Current Status: {pulp.LpStatus[prob.status]}")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+ds1-1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
--- Initial Feasibility Check ---
Total Supply Available: 248,756.40 Tons
Total Demand Required: 50,345.56 Tons

Total supply is GREATER than total demand. The problem should be able to meet demand, assuming sufficient capacities.
---------------------------------


--- Running the Optimization Solver ---
--- Solver Finished ---

Problem Status: Optimal

Optimal Total Transportation Cost: 933,252,049.40 Rwfs

Flows (Tons/Year):
  Kayonza_D -> Gasabo_Dem: 4,160.00 Tons (Cost: 19,100.00 Rwfs/T, UIJ: 4,160.00 Tons, LIJ: 54.50 Tons)
  Kayonza_D -> Kicukiro_Dem: 4,160.00 Tons (Cost: 17,760.00 Rwfs/T, UIJ: 4,160.00 Tons, LIJ: 58.60 Tons)
  Kayonza_D -> Kayonza_W2: 4,160.00 Tons (Cost: 1,540.00 Rwfs/T, UIJ: 4,160.00 Tons, LIJ: 675.30 Tons)
  Kayonza_D -> Nyarugenge_Dem: 4,160.00